In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import sys
from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch, Dataset
from tqdm import tqdm
import sys

sys.path.append("../")

DATA_DIR = "../mu3e_trigger_data"
MODEL_DIR = "../models"
PLOTS_DIR = "../plots"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/sig_with_layer_with_track_id_pixel_spacetime.npy"
BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/bg_with_layer_with_track_id_pixel_spacetime.npy"
SIGNAL_MPPC_FILE = f"{DATA_DIR}/sig_with_layer_with_track_id_mppc_spacetime.npy"
BACKGROUND_MPPC_FILE = f"{DATA_DIR}/bg_with_layer_with_track_id_mppc_spacetime.npy"

sig_only_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
sig_only_mppc_spacetime = np.load(SIGNAL_MPPC_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_MPPC_FILE)

X_pixel = np.concatenate([sig_only_pixel_spacetime, bg_pixel_spacetime], axis=0)
X_mppc = np.concatenate([sig_only_mppc_spacetime, bg_mppc_spacetime], axis=0)
y = np.concatenate(
    [np.ones(sig_only_pixel_spacetime.shape[0]), np.zeros(bg_pixel_spacetime.shape[0])], axis=0
)

from sklearn.model_selection import train_test_split
X_pixel_train, X_pixel_val, X_mppc_train, X_mppc_val, y_train, y_val = train_test_split(
    X_pixel, X_mppc, y, test_size=0.2, random_state=42, stratify=y
)

from src.torch.pre_processing.graph_batching import FullEventDataset, FullEventDataLoader
train_dataset = FullEventDataset(X_pixel_train, X_mppc_train, y_train, connect_layers=True)
val_dataset = FullEventDataset(X_pixel_val, X_mppc_val, y_val, connect_layers=True)
train_loader = FullEventDataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = FullEventDataLoader(val_dataset, batch_size=512, shuffle=False)



In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import global_mean_pool, global_max_pool
from torch_geometric.nn import GCNConv
from torch_geometric.nn import BatchNorm
from torch_scatter import scatter_mean, scatter_max
from src.torch.model.components import mlp


class SimpleGraphClassifier(nn.Module):
    """
    Simplified graph classifier that's easier to use and debug.
    """

    def __init__(
        self,
        node_input_dim: int,
        hidden_dim: int = 64,
        num_conv_layers: int = 4,
        output_dim: int = 32,
        dropout: float = 0.2,
    ):
        super().__init__()

        # Node feature processing
        self.node_projection = nn.Linear(node_input_dim, hidden_dim)

        # Graph convolution layers
        self.conv_layers = nn.ModuleList()
        self.batch_norms = nn.ModuleList()

        for _ in range(num_conv_layers):
            self.conv_layers.append(GCNConv(hidden_dim, hidden_dim))
            self.batch_norms.append(BatchNorm(hidden_dim))

        # Multi-scale pooling
        self.dropout = dropout

        self.feature_generaterator = mlp.get_mlp(
            input_dim=2 * hidden_dim,
            output_dim=output_dim,
            num_layers=4,
            dropout=dropout,
        )

        # Classifier

    def forward(
        self, x: torch.Tensor, edge_index: torch.Tensor, batch: torch.Tensor
    ) -> torch.Tensor:
        """
        Simplified forward pass.
        """
        # Project node features
        x = F.relu(self.node_projection(x))

        # Apply convolutions
        for conv, bn in zip(self.conv_layers, self.batch_norms):
            x_new = conv(x, edge_index)
            x_new = bn(x_new)
            x_new = F.relu(x_new)
            x_new = F.dropout(x_new, p=self.dropout, training=self.training)
            x = x + x_new  # Residual connection

        # Multi-scale pooling
        mean_pool = global_mean_pool(x, batch)
        max_pool = global_max_pool(x, batch)

        # Combine pooled features
        graph_features = torch.cat([mean_pool, max_pool], dim=-1)
        graph_features = self.feature_generaterator(graph_features)

        return graph_features


from torch_src.model.components import (
    get_mlp,
    TransformerBlock,
    SequenceGNNClassifier,
)


class SequenceClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=16, dropout=0.1):
        super(SequenceClassifier, self).__init__()
        self.transformer_block = TransformerBlock(
            input_dim, num_heads=4, dropout=dropout
        )
        self.classifier = get_mlp(2 * hidden_dim, 1, num_layers=3, dropout=dropout)

    def forward(self, x, batch_idx):
        pooling_mean = scatter_mean(x, batch_idx, dim=0)
        pooling_max = scatter_max(x, batch_idx, dim=0)[0]
        pooled = torch.cat([pooling_mean, pooling_max], dim=-1)
        out = self.classifier(pooled).squeeze(-1)
        return torch.sigmoid(out)


frame_classifier = SequenceGNNClassifier(
    gnn=SimpleGraphClassifier(
        node_input_dim=3, hidden_dim=32, num_conv_layers=4, output_dim=32, dropout=0.1
    ),
    classifier_module=SequenceClassifier(input_dim=32, hidden_dim=32, dropout=0.1),
)

In [ ]:
from torch.optim import AdamW
from torch_src.training import FocalLoss

optimizer = AdamW(frame_classifier.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = FocalLoss()

In [ ]:
from torcheval.metrics.aggregation import AUC
from torch_src.training import train_set_graph_classifier

train_set_graph_classifier(
    train_loader,
    val_loader,
    num_epochs=10,
    model=frame_classifier,
    criterion=criterion,
    MODEL_DIR=MODEL_DIR,
)

Model initialized with 23040 trainable parameters.


Epoch 1/10: 100%|██████████| 2192/2192 [01:57<00:00, 18.64it/s]


Epoch 1/10 - Training loss: 0.0821, AUC: 0.6625
Epoch 1/10 - Validation loss: 0.0796, AUC: 0.7165
Epoch 0: Train AUC: 0.6625, Val AUC: 0.7165


Epoch 2/10: 100%|██████████| 2192/2192 [02:13<00:00, 16.42it/s]


Epoch 2/10 - Training loss: 0.0796, AUC: 0.6942
Epoch 2/10 - Validation loss: 0.0776, AUC: 0.7270


Epoch 3/10: 100%|██████████| 2192/2192 [02:12<00:00, 16.54it/s]


Epoch 3/10 - Training loss: 0.0787, AUC: 0.7054
Epoch 3/10 - Validation loss: 0.0766, AUC: 0.7337


Epoch 4/10:  28%|██▊       | 608/2192 [00:36<01:34, 16.78it/s]